# para

> Paragraph rules: restatement, elegant variation, forced symmetry, and coinage

In [1]:
#| default_exp para

In [2]:
#| hide
from nbdev.showdoc import *

The rules that read a whole paragraph: restatement, elegant variation, forced symmetry, and coinage. The first two need a way to measure whether two different word sequences say the same thing, and word vectors provide it. The paragraph is the largest unit any of these rules analyzes, and coinage, which counts across paragraphs, does so with arithmetic alone.

In [3]:
#| export
from collections import Counter
from fastcore.utils import *
from slopometer.core import *
from slopometer.segment import *
from slopometer.syntax import *

In [4]:
from fastcore.test import *

## Word vectors in sixty seconds

A word vector is 300 numbers that summarize the contexts a word appears in, learned by counting co-occurrences over billions of words. Words used in similar contexts get similar vectors, and the cosine of the angle between two vectors is a workable measure of how related the words are: 1.0 means interchangeable contexts, near 0 means unrelated. The vectors in `en_core_web_md` come from the [GloVe](https://nlp.stanford.edu/projects/glove/) algorithm (Pennington, Socher, and Manning, 2014), trained on Common Crawl. Chapter 6 of [Jurafsky and Martin](https://web.stanford.edu/~jurafsky/slp3/) covers the theory from scratch.

Two properties matter for the rules below. Similarity is symmetric and graded, which lets a threshold say "these two words are close enough to be the same concept". And the vocabulary is finite: a token with no vector (`is_oov`) is a word the training corpus effectively never saw, which is how the coinage rule recognizes invented terms. spaCy averages token vectors to give spans and sentences a vector too, which is crude, order-blind, and still good enough to catch a sentence that restates its paragraph.

In [5]:
nlp = get_nlp()
v = nlp.vocab
[(a, b, round(v[a].similarity(v[b]), 3)) for a,b in [('kernel','interpreter'), ('folder','directory'), ('kernel','pineapple')]] + \
    [(w, 'oov', v[w].is_oov) for w in ('heartbeat', 'slopometer', 'exhash')]

[('kernel', 'interpreter', 0.383),
 ('folder', 'directory', 0.323),
 ('kernel', 'pineapple', 0.131),
 ('heartbeat', 'oov', False),
 ('slopometer', 'oov', True),
 ('exhash', 'oov', True)]

The numbers above are a warning as much as a demonstration. "folder" and "directory" reach only 0.32, and "kernel" and "interpreter" score 0.38, because GloVe learned from the general web, where a kernel is corn and an interpreter is a person, and md's pruned vector table even scrambles the ordering between such pairs. General-purpose vectors under-rate technical synonymy, so a rule that trusted a high similarity threshold to find technical word-swapping would miss most of it. The elegant-variation rule below therefore treats vectors as one signal of three, and the out-of-vocabulary flag, which needs no similarity at all, is exactly reliable for what the coinage rule needs: `slopometer` is a made-up word and the vocabulary knows it.

## Restatement

Tell 9's commonest form is a lead sentence that summarizes the paragraph it starts: delete it and nothing is lost. Averaged sentence vectors can measure this, but averaging flattens everything toward the middle, and every pair of sentences about the same software scores high. What separates restatement from progression is the margin above that baseline, and the margin has to be measured before a threshold means anything. One restating paragraph and one fact-adding paragraph, both in register:

In [6]:
restating = nlp("Failures are visible now too. A startup failure used to print to the server log and leave a half-booted dialog that looked ready. It now raises, and the user sees an error toast and a red status dot.")
progressing = nlp("The ready-wait runs once per kernel, in start. watch polls the process and the heartbeat. Three missed heartbeats mark the kernel unresponsive in its model.")

def _lead_sim(doc):
    sents = list(doc.sents)
    return round(sents[0].similarity(doc[sents[1].start:]), 3)

_lead_sim(restating), _lead_sim(progressing)

(0.857, 0.849)

The design fails, and the numbers above are the evidence: the restating and progressing paragraphs score within a hundredth of each other, and which one ranks higher flips between models. Averaged vectors measure shared topic, not shared claim, and every well-scoped paragraph shares its topic with its own lead. So lead-sentence restatement joins the unscoreable registry, with this measurement as the reason, rather than shipping as a rule that flags good paragraphs. What survives is the heading echo: a section whose first sentence repeats its heading's words ("## Configuration" followed by "Configuration is stored...") is detectable by plain lemma overlap, and the overlap is direct repetition, not paraphrase, so vectors are not needed at all.

This is the first rule that reads across blocks, and it sets the shape for document-level rules: they receive every block plus the parsed `Doc` for each prose block, aligned by position, and return findings with offsets already absolute.

In [7]:
#| export
def _content(toks): return {t.lemma_.lower() for t in toks if t.is_alpha and not t.is_stop}

@rule('heading_echo', tell=9, weight=SMELL, level='doc')
def find_heading_echo(blocks, docs):
    "A section's first sentence repeating its heading's words"
    res = []
    for b,nxt,d in zip(blocks, blocks[1:], docs[1:]):
        if b.kind != 'heading' or nxt.kind != 'prose' or d is None: continue
        hw = _content(get_nlp()(scrub(b.txt.lstrip('# '))))
        if not hw: continue
        s1 = list(d.sents)[0]
        lead = first((t.lemma_.lower() for t in s1 if t.is_alpha and not t.is_stop), None)
        if lead in hw and hw <= _content(s1): res.append(Finding('heading_echo', 9, nxt.start, nxt.start+len(s1.text), s1.text, SMELL))
    return res

In [8]:
#| export
def parse_blocks(blocks):
    "The spaCy `Doc` for each prose block, aligned with `blocks`, None elsewhere"
    prose = [b for b in blocks if b.kind=='prose']
    parsed = iter(get_nlp().pipe([scrub(b.txt) for b in prose]))
    return [next(parsed) if b.kind=='prose' else None for b in blocks]

In [9]:
echo_doc = "## Configuration\n\nConfiguration is stored in a single file.\n\n## Sockets\n\nEach worker owns one socket.\n"
bs = segment(echo_doc)
res = find_heading_echo(bs, parse_blocks(bs))
test_eq(len(res), 1)
res


[[3] heading_echo (tell 9, restatement): 'Configuration is stored in a single file.']

## Elegant variation

Fowler coined the name for the habit of dressing one concept in different words to avoid repetition: "the heartbeat" in one sentence becomes "beats" in the next, and the reader must decide whether two names mean two things. In reference prose the rule is STE's: one concept, one name, every time. In coding discussions the correct name is the code symbol when one exists: `restart` every time, never "the respawn operation" in one sentence and "recycling" in the next, and tell 5 makes the same demand of statuses and values. The tempting detector is vector similarity between distinct nouns in one paragraph, and it fails: cosine similarity measures shared context, and words that co-occur constantly score high whether or not they are interchangeable.

In [10]:
[(a, b, round(v[a].similarity(v[b]), 3)) for a,b in [('client','server'), ('read','write'), ('heartbeat','beat')]]

[('client', 'server', 0.272),
 ('read', 'write', 0.383),
 ('heartbeat', 'beat', 0.187)]

The measurements settle it: "client" and "server" score well above "heartbeat" and "beat", and a vector threshold would flag client-server paragraphs while missing the canonical variation pair. So vectors stay out of this rule. What remains is precise: a curated list of genuinely interchangeable term pairs, and stem containment, which catches one word embedded in the other ("beats" inside "heartbeats"). The rule flags the second name, which is where the variation happened. A token of one repeated character is excluded, because segmentation X-fills inline code and GloVe happens to know runs of x as real web tokens. The rule does not gate on syntactic role: in the `write_docs` example the two names appear as object and subject, so a role gate would miss the exact case the tell quotes.

In [11]:
#| export
_variant_pairs = [{'folder', 'directory'}, {'method', 'function'}, {'parameter', 'argument'}, {'error', 'exception'}, {'docstring', 'documentation'}]

@rule('variation', tell=4, weight=SMELL, level='para')
def find_variation(doc):
    "Two names for one concept within a paragraph, by curated pair or shared stem"
    seen,res,done = {},[],set()
    for t in doc:
        if t.pos_ not in ('NOUN', 'PROPN') or not t.is_alpha or len(set(t.lower_)) == 1: continue
        lem = t.lemma_.lower()
        for prev in seen:
            if prev == lem: continue
            pair = any(prev in s and lem in s for s in _variant_pairs)
            stem = (len(min(lem, prev, key=len)) >= 4 and (lem in prev or prev in lem)
                and not doc.vocab[lem].is_oov and not doc.vocab[prev].is_oov)
            if (pair or stem) and frozenset((prev, lem)) not in done:
                done.add(frozenset((prev, lem)))
                res.append(Finding('variation', 4, t.idx, t.idx+len(t.text), f'{seen[prev].text} ... {t.text}', SMELL))
                break
        seen[lem] = t
    return res

In [12]:
test_eq(find_variation(nlp('The heartbeat channel carries heartbeats only.')), [])
find_variation(nlp('Three missed heartbeats mark it unresponsive, and the next beat clears the mark. Put each parameter on its own line, and document every argument.'))

[[3] variation (tell 4, elegant variation): 'heartbeats ... beat',
 [3] variation (tell 4, elegant variation): 'parameter ... argument']

## Forced symmetry

Tell 20 is the AI habit of making material march in step: "fresh ports, fresh channels, and a fresh interpreter", three pros and three cons, every bullet opening with the same word. Real material is lumpy. Two symmetric shapes are mechanical enough to catch with precision. Inside a paragraph, three or more coordinated nouns each modified by the same adjective lemma is the marked example from the `write_docs` passage. Across a bullet list, three or more consecutive items opening with the same word is the list forced into one grammatical mold, and list items are their own blocks, so this half reads the document.

In [13]:
#| export
@rule('triad', tell=20, weight=SMELL, level='para')
def find_triad(doc):
    "Three or more coordinated nouns sharing one modifier lemma"
    res = []
    for t in doc:
        if t.pos_ != 'NOUN': continue
        group = [t] + [c for c in t.conjuncts if c.pos_ == 'NOUN']
        if len(group) < 3 or any(g.i < t.i for g in group): continue
        mods = [{m.lemma_ for m in g.children if m.dep_ == 'amod'} for g in group]
        shared = set.intersection(*mods) if all(mods) else set()
        if shared:
            span = doc.text[group[0].idx:group[-1].idx+len(group[-1].text)]
            res.append(Finding('triad', 20, group[0].idx, group[-1].idx+len(group[-1].text), span, SMELL))
    return res

@rule('bullet_mold', tell=20, weight=SMELL, level='doc')
def find_bullet_mold(blocks, docs):
    "Three or more consecutive list items opening with the same word"
    res,run = [],[]
    def word(b): return re.sub(r'^\s*(?:[-*+]|\d+[.)]) +', '', b.txt).split(' ')[0].lower() if b.txt else ''
    for b in list(blocks) + [None]:
        if b is not None and b.kind == 'item':
            run.append(b)
            continue
        if len(run) >= 3 and len({word(x) for x in run}) == 1:
            res.append(Finding('bullet_mold', 20, run[0].start, run[-1].start+len(run[-1].txt), '\n'.join(x.txt for x in run), SMELL))
        run = []
    return res

In [14]:
tri = find_triad(nlp('The kernel gains fresh ports, fresh channels, and a fresh interpreter.'))
bs2 = segment('- fast startup\n- fast shutdown\n- fast restarts\n')
tri + find_bullet_mold(bs2, parse_blocks(bs2))


[[3] triad (tell 20, forced symmetry): 'ports, fresh channels, and a fresh interpreter',
 [3] bullet_mold (tell 20, forced symmetry): '- fast startup\n- fast shutdown\n- fast restarts']

## The paragraph cap

ASD-STE100 caps paragraphs at six sentences. Counting sentences fights the sentence-length rule, though: splitting a long sentence raises the count, punishing the fix the meter itself demands. So the cap counts words instead, at 150, which is six sentences at the sentence cap, escalating per extra 25. An earlier version flagged the meter's own calibration passage, which was then a single nine-sentence paragraph. That conflict had two possible resolutions, and treating the fixture as untouchable would have picked one silently. Reading the passage settled it the other way: the paragraph really did pack a status contract and a restart story together, `write_docs` gained a paragraph break, and the rule ships. Nothing in this project is beyond refinement, including the calibration set, and this rule's history is the proof.


In [15]:
#| export
@rule('para_cap', tell=None, weight=PRESSURE, level='para')
def find_para_cap(doc):
    "Paragraphs past 150 words, escalating per extra 25"
    n = sum(1 for t in doc if t.is_alpha)
    if n <= 150: return []
    return [Finding('para_cap', None, 0, len(doc.text), f'{n}-word paragraph', PRESSURE*((n-150)//25 + 1))]


In [16]:
s25 = 'The gateway holds one socket open for every kernel it manages and closes each one when that kernel finally exits. '
test_eq(find_para_cap(nlp(s25 * 6)), [])
test_eq(find_para_cap(nlp('The server does a thing. ' * 9)), [])
find_para_cap(nlp(s25 * 8))


[[1] para_cap: '160-word paragraph']

## Coinage

Tell 12 has two halves, and one is definable: a locally-invented term dropped without definition. "The carrier" in a design doc cannot be looked up anywhere, and that is what separates a coinage from an established term of art. The vocabulary gives a mechanical proxy for "cannot be looked up": a word GloVe never saw in billions of web words (`is_oov`) is not established. The rule flags a lowercase out-of-vocabulary word that recurs outside backticks, and it flags the first occurrence, which is where the definition belonged. Backticked mentions are already invisible here because rules read scrubbed text, and that is the escape hatch working as designed: a coinage named as a code symbol is doing its job. Capitalized words stay out on purpose, since an out-of-vocabulary capitalized word is usually a product or a person, and both are names, not coinages.

In [17]:
#| export
@rule('coinage', tell=12, weight=SMELL, level='doc')
def find_coinage(blocks, docs):
    "A recurring out-of-vocabulary term outside backticks"
    cnt,firsts = Counter(),{}
    for b,d in zip(blocks, docs):
        if d is None: continue
        for t in d:
            if not t.is_alpha or len(t.text) < 4 or not t.text.islower() or not t.is_oov: continue
            cnt[t.text] += 1
            if t.text not in firsts: firsts[t.text] = (b.start+t.idx, b.start+t.idx+len(t.text))
    return [Finding('coinage', 12, *firsts[w], w, SMELL) for w,c in cnt.items() if c >= 2]

In [18]:
coin_doc = 'Each dlgpart holds one message. A dlgpart never owns its buffer.\n\nThe `exhash` addresses are separate, and `exhash` recurs freely in backticks.\n'
bs3 = segment(coin_doc)
res = find_coinage(bs3, parse_blocks(bs3))
test_eq(len(res), 1)
res

[[3] coinage (tell 12, audience misjudged): 'dlgpart']

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()